# 🧬 Vector Embeddings en Databricks — con la API de OpenAI y con SQL

## ¿Qué vamos a hacer en este notebook?

En este notebook vamos a aprender a **convertir texto en números** usando los **modelos de embeddings** que ya vienen alojados en Databricks. Lo haremos de **dos formas distintas** (las dos sirven, según el contexto):

1. **Con la API de OpenAI** (desde código Python) → ideal para aplicaciones.
2. **Con SQL** usando la función `ai_query` → ideal para procesar tablas enteras dentro del *lakehouse*.

> 🎯 **¿Por qué es tan importante este notebook?** Los *embeddings* son **el corazón de RAG** (*Retrieval-Augmented Generation*). Sin embeddings no hay búsqueda semántica, y sin búsqueda semántica no hay RAG. Todo lo que veremos en los siguientes notebooks (crear el índice vectorial, recuperar documentos, montar la aplicación) se apoya en lo que aprendas aquí.

---

### 🗺️ ¿Dónde encaja esto dentro del flujo de RAG?

RAG consiste en darle a un LLM **información extra y actualizada** en el momento de responder, buscándola primero en *tus* documentos. El flujo completo del curso es:

| Paso | Notebook | Qué hace |
|------|----------|----------|
| 1 | `Data_Prep` | Trocea (*chunking*) los documentos en fragmentos. |
| **2** | **`Vector_Embeddings` (este)** | **Convierte cada fragmento de texto en un vector numérico.** |
| 3 | `Vector_Index_Creation` | Guarda esos vectores en un índice para poder buscarlos. |
| 4 | `RAG_Application` | Busca los fragmentos relevantes y se los pasa al LLM. |
| 5 | `RAG_Evaluation` | Mide si las respuestas son buenas. |

> 📚 **Nota para el aprendizaje**: a lo largo del notebook verás celdas de texto (como esta) que explican el *qué* y el *por qué*, y celdas de código que ejecutan acciones. Lee siempre el texto antes de ejecutar el código de debajo.

### Installing Libraries and Utilities

In [ ]:
!pip install openai==1.69.0

### Creating an OpenAI Client

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
    base_url = "YOUR_DATABRICKS_WORKSPACE_HOSTNAME/serving-endpoints"
)

### Throwing a Request via the OpenAI API

In [ ]:
# OpenAI Request
completion = client.embeddings.create(
    model="databricks-gte-large-en",
    input="hi how are you?"
)

print("length \n")
print(len(completion.data[0].embedding))
print("\n embeddings: \n")
print(completion.data[0].embedding)


### Throwing Request via SQL Query

In [ ]:
%sql
SELECT ai_query(
    "databricks-gte-large-en",
    "Can you explain AI in ten words?"
  )